In [1]:
import zerorpc
vis = zerorpc.Client()
vis.connect("tcp://192.168.123.10:4242")

[<SocketContext(connect='tcp://192.168.123.10:4242')>]

In [2]:
import numpy as np
import zerorpc
import time
from dataclasses import dataclass
import threading

class NYUFingerHardware:
    def __init__(self, 
                 robot_ip = '192.168.123.10',
                #  q_offset = np.array([10.42,   1.023,  3.81]),
                 dt=0.01):
        self.robot = zerorpc.Client()
        self.robot.connect(f"tcp://{robot_ip}:4242")
        self.q_raw = np.zeros(3)
        self.q_dir = np.array([-1, 1, -1])
        self.dt = dt
        self.running = True
        self.alpha = 0.9
        self.dq_f = np.zeros(3) # filtered velocity
        # Reset the relative encocder value
        state = self.robot.getJointStates()
        self.q0_rel = np.array([state[f'joint_{i+1}']['q'] for i in range(3)])
        # Get the current absolute joint angles from the absolute joint encoders
        self.q_offset = self.getAbsoluteJointAngles()

    def getAbsoluteJointAngles(self):
        abs_state = self.robot.getAbsJointStates()
        q_abs_offset = np.array([10.4,   0.88,  3.84])
        q_abs = np.array([s['q_abs'] for s in abs_state.values()]) - q_abs_offset
        return q_abs*np.array([0.5,0.91, 1.])*np.array([-1, -1, -1])
    
    def getAbsoluteJointAnglesRaw(self):
        abs_state = self.robot.getAbsJointStates()
        q_abs = np.array([s['q_abs'] for s in abs_state.values()])
        return q_abs

    def get_state(self):
        state = self.robot.getJointStates()
        q = np.array([state[f'joint_{i+1}']['q'] for i in range(3)])
        dq = np.array([state[f'joint_{i+1}']['dq'] for i in range(3)])
        self.q_raw = q.copy()
        q = (q - self.q0_rel)*self.q_dir + self.q_offset
        dq = dq*self.q_dir
        self.dq_f = self.alpha*self.dq_f + (1-self.alpha)*dq
        return q, dq, self.dq_f # return joints position and filtered velocity
    
    def get_state_raw(self):
        state = self.robot.getJointStates()
        q = np.array([state[f'joint_{i+1}']['q'] for i in range(3)])
        dq = np.array([state[f'joint_{i+1}']['dq'] for i in range(3)])
        self.q_raw = q.copy()
        return q, dq, self.dq_f 
    
    def send_joint_cmd(self, q, dq, joint_torques):
        tauff = joint_torques*self.q_dir
        qdes = (q-self.q_offset)*self.q_dir + self.q0_rel #- self.q_offset
        dqdes = dq*self.q_dir
        command = {f'joint_{i+1}': {'q': qdes[i], 'dq': dqdes[i], 'tau': tauff[i]} for i in range(3)}
        self.robot.setJointCommand(command)

    def reset_sensors(self, q0=np.zeros(3)):
        pass

In [3]:
robot = NYUFingerHardware()

In [4]:
robot.get_state()

(array([-1.04528666, -0.16053914,  0.62372683]),
 array([ 0.00833333, -0.00222222, -0.005     ]),
 array([ 0.00083333, -0.00022222, -0.0005    ]))

In [9]:
robot.getAbsoluteJointAngles()

array([-0.68225827, -0.36982726, -0.13566829])

In [16]:
robot.send_joint_cmd(np.array([0.0,  0.0,  1.5666737]), np.array([0., 0., 0.]), np.array([0.0, 0.0, 0.0]))

In [ ]:
from NYUFinger.utils.vis import NYUFingerVisualizer
visulizer = NYUFingerVisualizer()

import time
for i in range(1000):
    q, dq, tau = robot.get_state()
    # q = robot.getAbsoluteJointAngles()
    visulizer.show(q)
    time.sleep(0.02)

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import time
from NYUFinger.real import NYUFingerHardware

robot_real = NYUFingerHardware(robot_ip='192.168.124.10', local_port=5001)

In [6]:
robot_real.reset_sensors()
q, dq = robot_real.get_state()
print(q)

Successfully reset the sensor values to: [0. 0. 0.]
[0. 0. 0.]


In [13]:
robot_real.get_state()

(array([-0.49082503,  0.21238195,  0.45647727]),
 array([-0.00568141,  0.        ,  0.00568141]))

In [18]:
robot.get_state()

(array([-0.15343388,  0.17236834, -1.26784352]),
 array([ 0.00833333,  0.00388889, -0.00222222]),
 array([ 0.00143333,  0.00100889, -0.00126222]))

In [21]:
import time

start_time = time.time()
while time.time()-start_time < 30:
    q, dq = robot_real.get_state()
    robot.send_joint_cmd(q, np.array([0., 0., 0.]), np.array([0.0, 0.0, 0.0]))
    robot_real.send_joint_torque(np.zeros(3))
    time.sleep(0.02)

In [19]:
import time
P = np.array([1.5, 1.5, 1.5])*100
# P = np.array([1.5, 1.5, 1.5])*100
D = np.array([0.05, 0.05, 0.05])
start_time = time.time()

counter = 0
while time.time()-start_time < 30:
    counter += 1
    q_des, dq_des, _ = robot.get_state()
    q, dq = robot_real.get_state()
    # follower control law
    error = q_des - q
    d_error = dq_des-dq 
    joint_torques = P * error + D * d_error
    robot_real.send_joint_torque(joint_torques)

    if counter%10==0:
        robot.send_joint_cmd(q, np.array([0., 0., 0.]), np.array([0.0, 0.0, 0.0]))

    time.sleep(0.001)

In [ ]:
q, dq, tau = robot.get_state()
for i in range(1000):
    q, dq, tau = robot.get_state()
    vis.show(q.tolist())
    time.sleep(0.05)